# Tasa de cambio representativa del mercado (TRM) desde 1991

La TRM es el precio oficial del dólar en pesos colombianos: el promedio ponderado de las compras y ventas de dólares del día. Es la serie de tiempo más usada del país y la que se trabaja en las clases del tema 2.4 (series de tiempo y ARIMA) y en la de LSTM.

|  |  |
|---|---|
| **Entidad** | Superintendencia Financiera de Colombia |
| **Conjunto** | Tasa de Cambio Representativa del Mercado – TRM · [`32sa-8pi3`](https://www.datos.gov.co/d/32sa-8pi3) |
| **Una fila es** | un periodo de vigencia de la tasa (un día hábil; la tasa del viernes rige también el fin de semana) |
| **Cobertura** | nacional, una sola serie |
| **Periodo** | 02/12/1991 a hoy |
| **Tamaño total** | 8.379 filas (consultado el 31/10/2026) |
| **Actualización** | diaria |
| **Requisitos** | ninguno |
| **Tiempo de descarga** | segundos |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base

Con `FRECUENCIA = "D"` el cuaderno expande cada vigencia a días calendario (todos los días tienen tasa). Con `"W"`, `"ME"` o `"YE"` se promedia por semana, mes o año.

| Columna | Qué es | Tipo |
|---|---|---|
| `fecha` | Día (o inicio de la semana, mes o año, según `FRECUENCIA`) | fecha |
| `trm` | Pesos por dólar (promedio del periodo si no es diaria) | número |
| `variacion_pct` | Cambio porcentual frente al periodo anterior (calculada) | número |
| `trm_min`, `trm_max` | Mínimo y máximo dentro del periodo (solo si no es diaria) | número |

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Pronóstico | `trm` del día o mes siguiente | rezagos de la propia serie (ARIMA, boosting con rezagos, LSTM) |
| Regresión | `variacion_pct` del mes siguiente | rezagos, petróleo o tasas de interés de otras fuentes |
| Clasificación | ¿Sube o baja la TRM el día siguiente? | rezagos y medias móviles |

**Cuidados antes de modelar**

- **Nunca baraje las filas:** parta en el tiempo (entrenamiento antes, prueba después) y use origen móvil o `TimeSeriesSplit`.
- Compare siempre contra la **línea base ingenua** (mañana = hoy): en la TRM es muy difícil de superar.
- Los rezagos y medias móviles deben calcularse solo con datos del pasado para no mirar el futuro.
- Con `FRECUENCIA = "D"` los fines de semana repiten la tasa del viernes: para modelar días hábiles use las filas originales.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

`FRECUENCIA` usa los códigos de pandas: `"D"` diaria, `"W"` semanal, `"ME"` mensual, `"YE"` anual.

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
FECHA_INI  = "2010-01-01"   # desde (la serie empieza el 1991-12-02)
FECHA_FIN  = None           # hasta, p. ej. "2025-12-31". None = hasta hoy
FRECUENCIA = "D"            # "D" diaria, "W" semanal, "ME" mensual, "YE" anual

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. Descarga

In [ ]:
TRM_ID = "32sa-8pi3"
filtros = [f"vigenciahasta >= '{FECHA_INI}T00:00:00'"]
if FECHA_FIN:
    filtros.append(f"vigenciadesde <= '{FECHA_FIN}T00:00:00'")
WHERE = " AND ".join(filtros)
print(f"Vigencias que cumplen el filtro: {entero(contar(TRM_ID, WHERE))}")
crudo = socrata(TRM_ID, where=WHERE, order="vigenciadesde")

## 5. De vigencias a días y a la frecuencia elegida

Cada vigencia se repite en todos los días que cubre (de `vigenciadesde` a `vigenciahasta`). Luego se recorta al rango pedido y se agrega.

In [ ]:
crudo["desde"] = pd.to_datetime(crudo["vigenciadesde"])
crudo["hasta"] = pd.to_datetime(crudo["vigenciahasta"])
crudo["trm"] = pd.to_numeric(crudo["valor"])
dias = (crudo.assign(fecha=[pd.date_range(d, h, freq="D") for d, h in zip(crudo["desde"], crudo["hasta"])])
             .explode("fecha")[["fecha", "trm"]])
dias["fecha"] = pd.to_datetime(dias["fecha"])
dias = dias.drop_duplicates("fecha", keep="last").set_index("fecha").sort_index()
dias = dias.loc[FECHA_INI:FECHA_FIN or None]

if FRECUENCIA == "D":
    trm = dias.reset_index()
else:
    trm = (dias["trm"].resample(FRECUENCIA)
               .agg(trm="mean", trm_min="min", trm_max="max").reset_index())
trm["variacion_pct"] = trm["trm"].pct_change() * 100
print(f"{entero(len(trm))} filas, del {trm['fecha'].min():%d/%m/%Y} al {trm['fecha'].max():%d/%m/%Y}")

## 6. Revisión de la base

In [ ]:
ficha_rapida(trm)
trm.tail()

*(Se completa al ejecutar.)*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
a1.plot(trm["fecha"], trm["trm"], color=AZUL, lw=1.2)
a1.yaxis.set_major_formatter(mtick.FuncFormatter(entero))
a1.set_title("TRM (pesos por dólar)")
a2.hist(trm["variacion_pct"].dropna(), bins=60, color=NARANJA, edgecolor="white", linewidth=0.4)
a2.set_title("Variación frente al periodo anterior (%)")
plt.tight_layout(); plt.show()
ingenuo = trm["trm"].diff().abs().mean()
print(f"Error medio del pronóstico ingenuo (mañana = hoy): {decimal(ingenuo)} pesos")

*(Se completa al ejecutar.)*

## 7. Guardar

In [ ]:
nombres = {"D": "diaria", "W": "semanal", "ME": "mensual", "YE": "anual"}
ruta = guardar(trm, etiqueta("trm", nombres.get(FRECUENCIA, FRECUENCIA)))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/trm_diaria.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.
5. Para un modelo con rezagos: `for k in range(1, 8): trm[f"lag_{k}"] = trm["trm"].shift(k)`.